# Chapter 6 (6.5–6.7): Data Storage, Reporting & End-to-End Pipeline

- **6.5** Data storage and retrieval strategies
- **6.6** Automated report generation
- **6.7** Case study: End-to-end automated analytics pipeline

> **Goal:** Build complete, production-ready analytics pipelines that store data efficiently and generate automated reports.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
import json
import logging
import time
import os
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Any

# For visualization in reports
import matplotlib.pyplot as plt
try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly not available. Install with: pip install plotly")

np.random.seed(42)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("AnalyticsPipeline")

---
## 6.5 Data Storage and Retrieval Strategies

### Storage Options Overview

| Storage Type | Best For | Examples |
|-------------|----------|----------|
| **Files (CSV/JSON)** | Small datasets, sharing | Local files, cloud storage |
| **Parquet/Feather** | Large datasets, fast reads | Data lakes, analytics |
| **SQL Databases** | Structured data, transactions | PostgreSQL, SQLite, MySQL |
| **NoSQL Databases** | Flexible schema, documents | MongoDB, Redis |
| **Data Warehouses** | Analytics, large scale | BigQuery, Snowflake, Redshift |

### 6.5.1 File-Based Storage

In [ ]:
# Create sample data
sales_data = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=1000, freq="H"),
    "product": np.random.choice(["Laptop", "Mouse", "Keyboard", "Monitor"], 1000),
    "quantity": np.random.randint(1, 50, 1000),
    "unit_price": np.random.choice([999.99, 29.99, 79.99, 299.99], 1000),
    "region": np.random.choice(["North", "South", "East", "West"], 1000)
})
sales_data["revenue"] = sales_data["quantity"] * sales_data["unit_price"]

print(f"Sample data: {len(sales_data)} rows")
sales_data.head()

In [ ]:
# Compare file formats
import os

# CSV
start = time.time()
sales_data.to_csv("sales.csv", index=False)
csv_write = time.time() - start
csv_size = os.path.getsize("sales.csv") / 1024  # KB

start = time.time()
df_csv = pd.read_csv("sales.csv")
csv_read = time.time() - start

# JSON
start = time.time()
sales_data.to_json("sales.json", orient="records")
json_write = time.time() - start
json_size = os.path.getsize("sales.json") / 1024

start = time.time()
df_json = pd.read_json("sales.json")
json_read = time.time() - start

# Parquet (if available)
try:
    start = time.time()
    sales_data.to_parquet("sales.parquet")
    parquet_write = time.time() - start
    parquet_size = os.path.getsize("sales.parquet") / 1024
    
    start = time.time()
    df_parquet = pd.read_parquet("sales.parquet")
    parquet_read = time.time() - start
except:
    parquet_write = parquet_read = parquet_size = None
    print("Parquet not available. Install with: pip install pyarrow")

# Comparison table
print("\n=== File Format Comparison ===")
print(f"{'Format':<10} {'Size (KB)':<12} {'Write (s)':<12} {'Read (s)':<12}")
print("-" * 46)
print(f"{'CSV':<10} {csv_size:<12.1f} {csv_write:<12.4f} {csv_read:<12.4f}")
print(f"{'JSON':<10} {json_size:<12.1f} {json_write:<12.4f} {json_read:<12.4f}")
if parquet_size:
    print(f"{'Parquet':<10} {parquet_size:<12.1f} {parquet_write:<12.4f} {parquet_read:<12.4f}")

**Discuss:**
- Which format is most space-efficient?
- Which format is fastest for reading?
- When would you choose CSV over Parquet?

### 6.5.2 SQL Database Storage

In [ ]:
# SQLite database example
db_path = "analytics.db"
conn = sqlite3.connect(db_path)

# Write data to database
start = time.time()
sales_data.to_sql("sales", conn, index=False, if_exists="replace")
db_write = time.time() - start

print(f"Wrote {len(sales_data)} rows to database in {db_write:.3f}s")

In [ ]:
# Query data from database
queries = {
    "all_data": "SELECT * FROM sales",
    "by_region": "SELECT region, SUM(revenue) as total FROM sales GROUP BY region",
    "top_products": "SELECT product, SUM(quantity) as units FROM sales GROUP BY product ORDER BY units DESC",
    "daily_sales": "SELECT DATE(date) as day, SUM(revenue) as daily_revenue FROM sales GROUP BY DATE(date)"
}

for name, query in queries.items():
    start = time.time()
    df = pd.read_sql(query, conn)
    elapsed = time.time() - start
    print(f"\n{name}: {len(df)} rows in {elapsed:.4f}s")
    print(df.head(3))

In [ ]:
# Parameterized queries (safe from SQL injection)
def get_sales_by_region(region: str) -> pd.DataFrame:
    """Get sales for a specific region."""
    query = "SELECT * FROM sales WHERE region = ?"
    return pd.read_sql(query, conn, params=[region])

def get_sales_in_range(start_date: str, end_date: str) -> pd.DataFrame:
    """Get sales within a date range."""
    query = "SELECT * FROM sales WHERE date BETWEEN ? AND ?"
    return pd.read_sql(query, conn, params=[start_date, end_date])

# Test parameterized queries
north_sales = get_sales_by_region("North")
print(f"North region sales: {len(north_sales)} rows")

jan_sales = get_sales_in_range("2024-01-01", "2024-01-15")
print(f"January 1-15 sales: {len(jan_sales)} rows")

### 6.5.3 Data Retrieval Strategies

In [ ]:
class DataStore:
    """
    A unified interface for data storage and retrieval.
    Supports multiple backends.
    """
    
    def __init__(self, storage_type: str = "sqlite", path: str = "data.db"):
        self.storage_type = storage_type
        self.path = path
        
        if storage_type == "sqlite":
            self.conn = sqlite3.connect(path)
    
    def save(self, df: pd.DataFrame, name: str, mode: str = "replace") -> None:
        """Save DataFrame to storage."""
        logger.info(f"Saving {len(df)} rows to {name}")
        
        if self.storage_type == "sqlite":
            df.to_sql(name, self.conn, index=False, if_exists=mode)
        elif self.storage_type == "parquet":
            df.to_parquet(f"{self.path}/{name}.parquet")
        elif self.storage_type == "csv":
            df.to_csv(f"{self.path}/{name}.csv", index=False)
    
    def load(self, name: str, query: str = None) -> pd.DataFrame:
        """Load DataFrame from storage."""
        if self.storage_type == "sqlite":
            if query:
                return pd.read_sql(query, self.conn)
            return pd.read_sql(f"SELECT * FROM {name}", self.conn)
        elif self.storage_type == "parquet":
            return pd.read_parquet(f"{self.path}/{name}.parquet")
        elif self.storage_type == "csv":
            return pd.read_csv(f"{self.path}/{name}.csv")
    
    def query(self, sql: str) -> pd.DataFrame:
        """Execute a SQL query (SQLite only)."""
        if self.storage_type != "sqlite":
            raise NotImplementedError("Query only supported for SQLite")
        return pd.read_sql(sql, self.conn)
    
    def close(self):
        """Close any open connections."""
        if hasattr(self, 'conn'):
            self.conn.close()

# Usage
store = DataStore("sqlite", "analytics.db")
store.save(sales_data, "sales_2024")

# Retrieve with query
top_regions = store.query("""
    SELECT region, SUM(revenue) as total_revenue
    FROM sales_2024
    GROUP BY region
    ORDER BY total_revenue DESC
""")
print(top_regions)

---
## 6.6 Automated Report Generation

### Report Types

| Type | Format | Use Case |
|------|--------|----------|
| **Text Reports** | Print, logs | Quick summaries, debugging |
| **HTML Reports** | Browser | Interactive, shareable |
| **PDF Reports** | Document | Formal reports, archival |
| **Excel Reports** | Spreadsheet | Business users, further analysis |
| **Dashboards** | Web app | Real-time monitoring |

### 6.6.1 Text Summary Reports

In [ ]:
def generate_text_report(df: pd.DataFrame, title: str = "Sales Report") -> str:
    """
    Generate a text-based summary report.
    """
    report = []
    report.append("=" * 60)
    report.append(f"  {title}")
    report.append(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append("=" * 60)
    
    # Key Metrics
    report.append("\n📊 KEY METRICS")
    report.append("-" * 40)
    report.append(f"  Total Revenue:     ${df['revenue'].sum():,.2f}")
    report.append(f"  Total Units Sold:  {df['quantity'].sum():,}")
    report.append(f"  Avg Order Value:   ${df['revenue'].mean():,.2f}")
    report.append(f"  Number of Orders:  {len(df):,}")
    
    # By Region
    report.append("\n🌍 REVENUE BY REGION")
    report.append("-" * 40)
    by_region = df.groupby("region")["revenue"].sum().sort_values(ascending=False)
    for region, revenue in by_region.items():
        pct = revenue / df['revenue'].sum() * 100
        report.append(f"  {region:<10} ${revenue:>12,.2f}  ({pct:.1f}%)")
    
    # By Product
    report.append("\n📦 TOP PRODUCTS")
    report.append("-" * 40)
    by_product = df.groupby("product").agg({
        "revenue": "sum",
        "quantity": "sum"
    }).sort_values("revenue", ascending=False)
    
    for product, row in by_product.iterrows():
        report.append(f"  {product:<12} ${row['revenue']:>10,.2f}  ({row['quantity']:,} units)")
    
    report.append("\n" + "=" * 60)
    
    return "\n".join(report)

# Generate and display report
text_report = generate_text_report(sales_data, "Q1 2024 Sales Report")
print(text_report)

In [ ]:
# Save report to file
with open("sales_report.txt", "w") as f:
    f.write(text_report)
print("Report saved to sales_report.txt")

### 6.6.2 HTML Reports with Visualizations

In [ ]:
def generate_html_report(df: pd.DataFrame, title: str = "Analytics Report") -> str:
    """
    Generate an HTML report with tables and embedded charts.
    """
    # Calculate metrics
    total_revenue = df['revenue'].sum()
    total_units = df['quantity'].sum()
    avg_order = df['revenue'].mean()
    
    by_region = df.groupby("region")["revenue"].sum().reset_index()
    by_product = df.groupby("product").agg({
        "revenue": "sum",
        "quantity": "sum"
    }).reset_index()
    
    # Create daily trend
    df_copy = df.copy()
    df_copy['day'] = pd.to_datetime(df_copy['date']).dt.date
    daily = df_copy.groupby('day')['revenue'].sum().reset_index()
    
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>{title}</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
            .container {{ max-width: 1200px; margin: 0 auto; background: white; padding: 30px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
            h1 {{ color: #333; border-bottom: 2px solid #4CAF50; padding-bottom: 10px; }}
            h2 {{ color: #555; margin-top: 30px; }}
            .metrics {{ display: flex; gap: 20px; margin: 20px 0; }}
            .metric-card {{ background: #f8f9fa; padding: 20px; border-radius: 8px; flex: 1; text-align: center; border-left: 4px solid #4CAF50; }}
            .metric-value {{ font-size: 24px; font-weight: bold; color: #333; }}
            .metric-label {{ color: #666; margin-top: 5px; }}
            table {{ width: 100%; border-collapse: collapse; margin: 20px 0; }}
            th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
            th {{ background: #4CAF50; color: white; }}
            tr:hover {{ background: #f5f5f5; }}
            .timestamp {{ color: #999; font-size: 12px; margin-top: 20px; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>📊 {title}</h1>
            
            <div class="metrics">
                <div class="metric-card">
                    <div class="metric-value">${total_revenue:,.0f}</div>
                    <div class="metric-label">Total Revenue</div>
                </div>
                <div class="metric-card">
                    <div class="metric-value">{total_units:,}</div>
                    <div class="metric-label">Units Sold</div>
                </div>
                <div class="metric-card">
                    <div class="metric-value">${avg_order:,.0f}</div>
                    <div class="metric-label">Avg Order</div>
                </div>
                <div class="metric-card">
                    <div class="metric-value">{len(df):,}</div>
                    <div class="metric-label">Total Orders</div>
                </div>
            </div>
            
            <h2>🌍 Revenue by Region</h2>
            <table>
                <tr><th>Region</th><th>Revenue</th><th>% of Total</th></tr>
    """
    
    for _, row in by_region.iterrows():
        pct = row['revenue'] / total_revenue * 100
        html += f"<tr><td>{row['region']}</td><td>${row['revenue']:,.2f}</td><td>{pct:.1f}%</td></tr>"
    
    html += """
            </table>
            
            <h2>📦 Product Performance</h2>
            <table>
                <tr><th>Product</th><th>Revenue</th><th>Units Sold</th></tr>
    """
    
    for _, row in by_product.iterrows():
        html += f"<tr><td>{row['product']}</td><td>${row['revenue']:,.2f}</td><td>{row['quantity']:,}</td></tr>"
    
    html += f"""
            </table>
            
            <p class="timestamp">Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        </div>
    </body>
    </html>
    """
    
    return html

# Generate and save HTML report
html_report = generate_html_report(sales_data, "Q1 2024 Sales Report")
with open("sales_report.html", "w") as f:
    f.write(html_report)
print("HTML report saved to sales_report.html")

### 6.6.3 Excel Reports

In [ ]:
def generate_excel_report(df: pd.DataFrame, filename: str = "report.xlsx") -> None:
    """
    Generate an Excel report with multiple sheets.
    """
    try:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            # Raw data
            df.to_excel(writer, sheet_name='Raw Data', index=False)
            
            # Summary by region
            by_region = df.groupby("region").agg({
                "revenue": ["sum", "mean", "count"],
                "quantity": "sum"
            }).round(2)
            by_region.columns = ['Total Revenue', 'Avg Revenue', 'Order Count', 'Units Sold']
            by_region.to_excel(writer, sheet_name='By Region')
            
            # Summary by product
            by_product = df.groupby("product").agg({
                "revenue": ["sum", "mean"],
                "quantity": "sum"
            }).round(2)
            by_product.columns = ['Total Revenue', 'Avg Revenue', 'Units Sold']
            by_product.to_excel(writer, sheet_name='By Product')
            
            # Daily summary
            df_copy = df.copy()
            df_copy['date'] = pd.to_datetime(df_copy['date']).dt.date
            daily = df_copy.groupby('date').agg({
                "revenue": "sum",
                "quantity": "sum"
            }).round(2)
            daily.to_excel(writer, sheet_name='Daily Trend')
        
        print(f"Excel report saved to {filename}")
        
    except ImportError:
        print("openpyxl not installed. Run: pip install openpyxl")

generate_excel_report(sales_data, "sales_report.xlsx")

### 6.6.4 Automated Report Scheduler

In [ ]:
class ReportGenerator:
    """
    Automated report generator with multiple output formats.
    """
    
    def __init__(self, data_store: DataStore, output_dir: str = "reports"):
        self.data_store = data_store
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
    
    def generate(self, query: str, report_name: str, formats: List[str] = None) -> Dict[str, str]:
        """
        Generate reports in specified formats.
        """
        formats = formats or ["txt", "html"]
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Fetch data
        logger.info(f"Generating report: {report_name}")
        df = self.data_store.query(query)
        logger.info(f"Fetched {len(df)} rows")
        
        output_files = {}
        
        for fmt in formats:
            filename = self.output_dir / f"{report_name}_{timestamp}.{fmt}"
            
            if fmt == "txt":
                content = generate_text_report(df, report_name)
                with open(filename, "w") as f:
                    f.write(content)
            
            elif fmt == "html":
                content = generate_html_report(df, report_name)
                with open(filename, "w") as f:
                    f.write(content)
            
            elif fmt == "csv":
                df.to_csv(filename, index=False)
            
            output_files[fmt] = str(filename)
            logger.info(f"Generated {fmt.upper()}: {filename}")
        
        return output_files

# Usage example
report_gen = ReportGenerator(store, "reports")
files = report_gen.generate(
    query="SELECT * FROM sales_2024",
    report_name="Daily_Sales",
    formats=["txt", "html", "csv"]
)
print(f"\nGenerated files: {files}")

**Try:**
1. Add PDF generation (requires `weasyprint` or `fpdf`)
2. Add email notification when reports are generated
3. Create a weekly report that compares to previous week

---
## 6.7 Case Study: End-to-End Automated Analytics Pipeline

Let's build a complete pipeline that:
1. **Extracts** data from multiple sources
2. **Transforms** and validates data
3. **Loads** to a data warehouse
4. **Generates** automated reports
5. **Monitors** pipeline health

In [ ]:
class AnalyticsPipeline:
    """
    End-to-end analytics pipeline with ETL, storage, and reporting.
    """
    
    def __init__(self, name: str, db_path: str = "analytics_pipeline.db"):
        self.name = name
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path)
        self.logger = logging.getLogger(name)
        self.metrics = {
            "runs": 0,
            "success": 0,
            "failures": 0,
            "rows_processed": 0,
            "last_run": None
        }
    
    # ===== EXTRACT =====
    def extract_sales(self) -> pd.DataFrame:
        """Extract sales data (simulated)."""
        self.logger.info("Extracting sales data...")
        
        data = pd.DataFrame({
            "date": pd.date_range("2024-01-01", periods=500, freq="H"),
            "product": np.random.choice(["Laptop", "Mouse", "Keyboard", "Monitor"], 500),
            "quantity": np.random.randint(1, 30, 500),
            "unit_price": np.random.choice([999.99, 29.99, 79.99, 299.99], 500),
            "region": np.random.choice(["North", "South", "East", "West"], 500),
            "customer_id": np.random.randint(1000, 9999, 500)
        })
        
        # Add some data quality issues for realistic testing
        data.loc[10:15, "quantity"] = np.nan
        data.loc[20, "unit_price"] = -99.99
        
        self.logger.info(f"Extracted {len(data)} sales records")
        return data
    
    def extract_customers(self) -> pd.DataFrame:
        """Extract customer data (simulated)."""
        self.logger.info("Extracting customer data...")
        
        customer_ids = list(range(1000, 10000))
        np.random.shuffle(customer_ids)
        
        data = pd.DataFrame({
            "customer_id": customer_ids[:200],
            "name": [f"Customer_{i}" for i in range(200)],
            "segment": np.random.choice(["Premium", "Standard", "Basic"], 200),
            "signup_date": pd.date_range("2020-01-01", periods=200, freq="5D")
        })
        
        self.logger.info(f"Extracted {len(data)} customer records")
        return data
    
    # ===== TRANSFORM =====
    def transform_clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Clean and validate data."""
        self.logger.info("Cleaning data...")
        df = df.copy()
        
        initial_rows = len(df)
        
        # Handle nulls
        if "quantity" in df.columns:
            null_count = df["quantity"].isnull().sum()
            if null_count > 0:
                self.logger.warning(f"Found {null_count} null quantities, filling with median")
                df["quantity"] = df["quantity"].fillna(df["quantity"].median())
        
        # Remove invalid prices
        if "unit_price" in df.columns:
            invalid = (df["unit_price"] <= 0).sum()
            if invalid > 0:
                self.logger.warning(f"Removing {invalid} rows with invalid prices")
                df = df[df["unit_price"] > 0]
        
        self.logger.info(f"Cleaned: {initial_rows} → {len(df)} rows")
        return df
    
    def transform_enrich(self, sales: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
        """Enrich sales with customer data."""
        self.logger.info("Enriching sales data...")
        
        # Calculate revenue
        sales["revenue"] = sales["quantity"] * sales["unit_price"]
        
        # Add time dimensions
        sales["date"] = pd.to_datetime(sales["date"])
        sales["year"] = sales["date"].dt.year
        sales["month"] = sales["date"].dt.month
        sales["day_of_week"] = sales["date"].dt.day_name()
        
        # Join with customer data
        enriched = sales.merge(customers[["customer_id", "segment"]], 
                               on="customer_id", 
                               how="left")
        enriched["segment"] = enriched["segment"].fillna("Unknown")
        
        self.logger.info(f"Enriched data: {len(enriched)} rows, {len(enriched.columns)} columns")
        return enriched
    
    # ===== LOAD =====
    def load(self, df: pd.DataFrame, table_name: str) -> None:
        """Load data to database."""
        self.logger.info(f"Loading {len(df)} rows to {table_name}...")
        df.to_sql(table_name, self.conn, index=False, if_exists="replace")
        self.logger.info(f"Loaded successfully")
    
    # ===== REPORT =====
    def generate_report(self) -> str:
        """Generate analytics report."""
        self.logger.info("Generating report...")
        
        # Fetch aggregated data
        summary = pd.read_sql("""
            SELECT 
                COUNT(*) as total_orders,
                SUM(revenue) as total_revenue,
                AVG(revenue) as avg_order_value,
                SUM(quantity) as total_units
            FROM sales_enriched
        """, self.conn).iloc[0]
        
        by_segment = pd.read_sql("""
            SELECT segment, SUM(revenue) as revenue
            FROM sales_enriched
            GROUP BY segment
            ORDER BY revenue DESC
        """, self.conn)
        
        by_region = pd.read_sql("""
            SELECT region, SUM(revenue) as revenue
            FROM sales_enriched
            GROUP BY region
            ORDER BY revenue DESC
        """, self.conn)
        
        # Build report
        report = []
        report.append("\n" + "=" * 60)
        report.append(f"  📊 ANALYTICS PIPELINE REPORT")
        report.append(f"  Pipeline: {self.name}")
        report.append(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append("=" * 60)
        
        report.append("\n📈 KEY METRICS")
        report.append("-" * 40)
        report.append(f"  Total Orders:    {summary['total_orders']:,.0f}")
        report.append(f"  Total Revenue:   ${summary['total_revenue']:,.2f}")
        report.append(f"  Avg Order Value: ${summary['avg_order_value']:,.2f}")
        report.append(f"  Total Units:     {summary['total_units']:,.0f}")
        
        report.append("\n👥 REVENUE BY CUSTOMER SEGMENT")
        report.append("-" * 40)
        for _, row in by_segment.iterrows():
            report.append(f"  {row['segment']:<15} ${row['revenue']:>12,.2f}")
        
        report.append("\n🌍 REVENUE BY REGION")
        report.append("-" * 40)
        for _, row in by_region.iterrows():
            report.append(f"  {row['region']:<10} ${row['revenue']:>12,.2f}")
        
        report.append("\n" + "=" * 60)
        
        return "\n".join(report)
    
    # ===== RUN PIPELINE =====
    def run(self) -> Dict[str, Any]:
        """
        Execute the complete analytics pipeline.
        """
        start_time = datetime.now()
        self.metrics["runs"] += 1
        
        self.logger.info("\n" + "#" * 60)
        self.logger.info(f"# STARTING PIPELINE: {self.name}")
        self.logger.info("#" * 60)
        
        try:
            # Extract
            self.logger.info("\n[1/4] EXTRACT PHASE")
            sales = self.extract_sales()
            customers = self.extract_customers()
            
            # Transform
            self.logger.info("\n[2/4] TRANSFORM PHASE")
            sales_clean = self.transform_clean(sales)
            sales_enriched = self.transform_enrich(sales_clean, customers)
            
            # Load
            self.logger.info("\n[3/4] LOAD PHASE")
            self.load(sales_enriched, "sales_enriched")
            self.load(customers, "customers")
            
            # Report
            self.logger.info("\n[4/4] REPORT PHASE")
            report = self.generate_report()
            print(report)
            
            # Update metrics
            self.metrics["success"] += 1
            self.metrics["rows_processed"] += len(sales_enriched)
            self.metrics["last_run"] = start_time
            
            duration = (datetime.now() - start_time).total_seconds()
            
            self.logger.info("\n" + "#" * 60)
            self.logger.info(f"# PIPELINE COMPLETE: {duration:.2f} seconds")
            self.logger.info("#" * 60)
            
            return {
                "status": "success",
                "duration": duration,
                "rows_processed": len(sales_enriched)
            }
            
        except Exception as e:
            self.metrics["failures"] += 1
            self.logger.error(f"Pipeline failed: {e}")
            return {
                "status": "failed",
                "error": str(e)
            }
    
    def get_metrics(self) -> Dict[str, Any]:
        """Get pipeline metrics."""
        return self.metrics.copy()

In [ ]:
# Create and run the pipeline
pipeline = AnalyticsPipeline("Sales Analytics v1")
result = pipeline.run()

print(f"\n=== PIPELINE RESULT ===")
print(f"Status: {result['status']}")
if result['status'] == 'success':
    print(f"Duration: {result['duration']:.2f}s")
    print(f"Rows Processed: {result['rows_processed']}")

In [ ]:
# View pipeline metrics
metrics = pipeline.get_metrics()
print("\n=== PIPELINE METRICS ===")
for key, value in metrics.items():
    print(f"{key}: {value}")

---
## Mini Project: Extend the Pipeline (20-30 min)

### Challenge
Extend the `AnalyticsPipeline` class with:

1. **Data Validation**: Add checks for data quality before loading
2. **Incremental Loading**: Only process new records
3. **Email Alerts**: Send notification on failure (simulate with print)
4. **Visualization**: Add a chart to the report
5. **Scheduling**: Add a method to run on a schedule

### Starter Extensions

In [ ]:
# Your extended pipeline code here

class ExtendedPipeline(AnalyticsPipeline):
    """
    Extended pipeline with additional features.
    """
    
    def validate_data(self, df: pd.DataFrame) -> bool:
        """Validate data quality before loading."""
        # TODO: Implement validation checks
        pass
    
    def send_alert(self, message: str) -> None:
        """Send alert on pipeline failure."""
        # TODO: Implement alert (print for now)
        pass
    
    # Add more methods...

---
## Summary

### Key Concepts Covered

| Topic | Key Takeaway |
|-------|-------------|
| **Storage Strategies** | Choose format based on size, speed, and use case |
| **SQL Databases** | Best for structured data with complex queries |
| **Automated Reports** | Generate consistent reports in multiple formats |
| **End-to-End Pipelines** | Combine ETL, storage, and reporting in one system |

### Best Practices

1. **Use Parquet** for large datasets (smaller, faster)
2. **Validate data** before loading to catch issues early
3. **Log everything** for debugging and auditing
4. **Automate reports** to save time and ensure consistency
5. **Monitor pipelines** to catch failures quickly